# 第 22 章互动实验：能不能看见这个缺陷

用你的成像仿真器回答一个设计问题：在给定光学和曝光下，多大的缺陷能被检出。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "22_成像与设备仿真"
ASSIGNMENT = "22-1_成像仿真.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
for lam, na in [(532, 0.5), (532, 0.9), (266, 0.9), (193, 0.95)]:
    r = attempt(mod.rayleigh_resolution_um, lam, na)
    if r is None: break
    print(f"λ={lam} nm, NA={na}：分辨率 {r:.3f} µm，建议像素（物面）≤ {r / 2:.3f} µm")

## 实验 1：缺陷信噪比 vs 缺陷尺寸

场景：灰度 0.5 的平坦背景上一个亮颗粒。PSF sigma 固定，改变颗粒直径和曝光量。

In [ ]:
def defect_snr(diam_px, full_scale_e, psf_sigma=1.5, contrast=0.3, trials=30):
    y, x = np.mgrid[0:41, 0:41]
    scene = 0.5 + contrast * (((y - 20) ** 2 + (x - 20) ** 2) <= (diam_px / 2) ** 2)
    k = mod.gaussian_psf(psf_sigma)
    sharp = mod.convolve_same(scene, k)
    bg = mod.convolve_same(np.full_like(scene, 0.5), k)
    rng = np.random.default_rng(0)
    diffs = []
    for _ in range(trials):
        a = mod.simulate_capture(sharp * full_scale_e, 5.0, 2.0, 12, rng).astype(float)
        b = mod.simulate_capture(bg * full_scale_e, 5.0, 2.0, 12, rng).astype(float)
        diffs.append((a - b)[20, 20])
    noise = np.std([mod.simulate_capture(bg * full_scale_e, 5.0, 2.0, 12, rng)[20, 20] for _ in range(200)]) * np.sqrt(2)
    return np.mean(diffs) / noise

diams = [0.5, 1, 1.5, 2, 3, 4, 6]
for fs in (500, 2000, 6000):
    snr = attempt(lambda: [defect_snr(d, fs) for d in diams])
    if snr is None: break
    plt.plot(diams, snr, "o-", label=f"满量程 {fs} e-")
else:
    plt.axhline(5, color="k", ls="--", label="SNR=5 参考线")
    plt.xlabel("缺陷直径 (px)"); plt.ylabel("中心像素信噪比"); plt.legend(); plt.show()

## 实验 2：光子转移曲线

In [ ]:
rng = np.random.default_rng(3)
levels = np.array([20, 50, 100, 300, 800, 1500, 3000, 6000, 12000, 20000], float)
means, vars_ = [], []
for lv in levels:
    f = attempt(mod.simulate_capture, np.full((100, 100), lv), 8.0, 1.5, 12, rng)
    if f is None: break
    means.append(f.mean()); vars_.append(f.var())
if len(means) == len(levels):
    plt.loglog(means, vars_, "o-"); plt.xlabel("均值 (ADU)"); plt.ylabel("方差 (ADU²)"); plt.title("PTC：注意高端的饱和弯折"); plt.show()
    ok = np.array(means) < 0.7 * 4095
    print("只用未饱和点拟合：", mod.estimate_gain_from_ptc(np.array(means)[ok], np.array(vars_)[ok]), "真值 (1.5, 8.0)")

## 破坏实验与记录

- 用全部点（含饱和点）拟合 PTC，增益估计偏向哪边？
- 把 PSF sigma 从 1.5 改成 3（离焦），实验 1 的曲线怎样？
- 写下你的结论：在你设定的参数下最小可检出缺陷多大，依据是什么。